# Bronze Transformations

## Objetivo
Este notebook tem como objetivo transformar os dados da camada `01_raw` em tabelas estruturadas na camada `02_bronze`, aplicando tratamentos técnicos mínimos para padronização e preparação das bases.

## Escopo da camada bronze
Nesta etapa, serão realizados apenas tratamentos técnicos, tais como:
- padronização de nomes de colunas
- ajuste de tipos de dados
- normalização estrutural de arquivos JSON
- remoção de duplicidades quando aplicável
- padronização técnica de campos categóricos essenciais

## Tabelas processadas
- `lh_nautical.01_raw.vendas_raw`
- `lh_nautical.01_raw.produtos_raw`
- `lh_nautical.01_raw.clientes_raw`
- `lh_nautical.01_raw.custos_importacao_raw`

## Saídas esperadas
- `lh_nautical.02_bronze.vendas_bronze`
- `lh_nautical.02_bronze.produtos_bronze`
- `lh_nautical.02_bronze.clientes_bronze`
- `lh_nautical.02_bronze.custos_importacao_bronze`

## Observação
Regras de negócio, joins entre entidades, métricas analíticas e tabelas finais para consumo serão tratados nas camadas `03_silver` e `04_gold` com auxilio do dbt.

In [0]:
# ======================================================================================
# IMPORTS
# ======================================================================================

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType, IntegerType, DateType
import unicodedata

In [0]:
# ======================================================================================
# SESSÃO SPARK
# ======================================================================================


spark = SparkSession.builder.getOrCreate()

## Funções auxiliares

As funções abaixo iniciam a padronizam nomes de colunas e categorias de produtos.

In [0]:
# ======================================================================================
# FUNÇÕES AUXILIARES
# ======================================================================================

def standardize_column_name(column_name: str) -> str:
    """
    Padroniza nomes de colunas para snake_case simples, em minúsculas.

    Regras aplicadas:
    - remove espaços nas extremidades
    - converte para minúsculas
    - substitui espaços por underscore
    - remove caracteres especiais básicos
    """
    cleaned = column_name.strip().lower()
    cleaned = cleaned.replace(" ", "_")
    cleaned = cleaned.replace("-", "_")
    cleaned = cleaned.replace("/", "_")
    cleaned = cleaned.replace("(", "")
    cleaned = cleaned.replace(")", "")
    cleaned = cleaned.replace(".", "")
    return cleaned


def normalize_text(text: str) -> str:
    """
    Remove acentos e padroniza texto para comparação.
    Retorna string em minúsculas e sem espaços excedentes.
    """
    if text is None:
        return None
    
    text = str(text).strip().lower()
    text = unicodedata.normalize("NFKD", text)
    text = "".join([c for c in text if not unicodedata.combining(c)])
    return text


def normalize_category(category: str) -> str:
    """
    Padroniza categorias de produtos para os três valores de negócio esperados:
    - eletronicos
    - propulsao
    - ancoragem

    Mantém None se a categoria estiver ausente.
    """
    if category is None:
        return None

    category_norm = normalize_text(category)

    if "eletr" in category_norm:
        return "eletronicos"
    elif "prop" in category_norm:
        return "propulsao"
    elif "ancor" in category_norm:
        return "ancoragem"
    else:
        return category_norm

In [0]:
# ======================================================================================
# REGISTRO DE UDF
# ======================================================================================
# A função Python de normalização de categoria é registrada como UDF para aplicação
# sobre DataFrames Spark.
# ======================================================================================

normalize_category_udf = F.udf(normalize_category)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/udf.py:103: UserWarning: Cannot infer the eval type from type hints. 
  warnings.warn("Cannot infer the eval type from type hints. ", UserWarning)


In [0]:
# ======================================================================================
# FUNÇÃO DE PADRONIZAÇÃO DE COLUNAS
# ======================================================================================

def standardize_dataframe_columns(df):
    """
    Renomeia todas as colunas do DataFrame para um padrão técnico consistente.
    """
    renamed_columns = [standardize_column_name(col_name) for col_name in df.columns]
    return df.toDF(*renamed_columns)

## 1. Transformação da tabela de produtos

Nesta etapa serão aplicados:
- padronização dos nomes de colunas
- normalização final das categorias
- conversão de colunas numéricas
- remoção de duplicidades

In [0]:
# ======================================================================================
# LEITURA - PRODUTOS RAW
# ======================================================================================

df_produtos_raw = spark.table("lh_nautical.01_raw.produtos_raw")
print("Quantidade de registros em produtos_raw:", df_produtos_raw.count())
display(df_produtos_raw.limit(10))

Quantidade de registros em produtos_raw: 157


name,price,code,actual_category
Transponder AIS Maré Magnum,R$ 33122.52,1,ELETRONICOS
Transponder Furuno Marlin,R$ 13998.15,2,ELETRONICOS
Radar Furuno Pulse Leviathan,R$ 9024.19,3,E L E T R Ô N I C O S
Rádio AIS Hydro Tidal Zen,R$ 3381.88,4,Eletrunicos
Piloto Automático Furuno Storm,R$ 23669.01,5,Eletronicoz
Transponder AIS Vector,R$ 11820.21,6,Eletrunicos
Radar AIS Zen,R$ 19518.77,7,eLeTrÔnIcOs
GPS AIS Zen,R$ 4984.15,8,E L E T R Ô N I C O S
Transponder AIS Titan Pulse,R$ 39705.5,9,Eletronicoz
Piloto Automático Simrad Titan Flux Magnum,R$ 32033.04,10,eletrônicos


In [0]:
# ======================================================================================
# CONTINUAÇÃO DA TRANSFORMAÇÃO - PRODUTOS BRONZE
# ======================================================================================


df_produtos_bronze = standardize_dataframe_columns(df_produtos_raw)

# Criação da coluna category padronizada a partir da coluna original actual_category
if "actual_category" in df_produtos_bronze.columns:
    df_produtos_bronze = df_produtos_bronze.withColumn(
        "category",
        normalize_category_udf(F.col("actual_category"))
    )

# Limpeza de campos monetários antes da conversão para double
monetary_columns = ["price", "cost"]

for col_name in monetary_columns:
    if col_name in df_produtos_bronze.columns:
        df_produtos_bronze = df_produtos_bronze.withColumn(
            col_name,
            F.regexp_replace(F.col(col_name), r"R\$\s*", "")
        )
        df_produtos_bronze = df_produtos_bronze.withColumn(
            col_name,
            F.regexp_replace(F.col(col_name), ",", ".")
        )

# Conversão de tipos numéricos
numeric_cast_map = {
    "code": IntegerType(),
    "price": DoubleType(),
    "cost": DoubleType(),
    "stock": IntegerType()
}

for col_name, col_type in numeric_cast_map.items():
    if col_name in df_produtos_bronze.columns:
        df_produtos_bronze = df_produtos_bronze.withColumn(
            col_name,
            F.col(col_name).cast(col_type)
        )

# Remoção de duplicidades exatas
total_produtos_antes = df_produtos_bronze.count()

df_produtos_bronze = df_produtos_bronze.dropDuplicates()

total_produtos_depois = df_produtos_bronze.count()
duplicados_removidos_produtos = total_produtos_antes - total_produtos_depois

print("Duplicados removidos em produtos:", duplicados_removidos_produtos)

display(df_produtos_bronze.limit(10))

Duplicados removidos em produtos: 0


name,price,code,actual_category,category
Rádio Furuno Zen Swift,35811.08,14,Eletroniscos,eletronicos
Sonda Lowrance Tidal Storm Vox,16432.35,19,eletronicos,eletronicos
Motor de Popa Volvo Hydro Dash 256HP,99664.64,72,Propução,propulsao
Motor Elétrico Honda Mako Axis 131HP,121761.56,78,propulsão,propulsao
Motor de Popa Parsun Impulse 162HP,38578.7,85,P R O P U L S Ã O,p r o p u l s a o
Corrente Danforth Zen Abyss Axis,4103.01,117,ANCORAGEM,ancoragem
Cabo de Nylon Delta Velocity Core Mako,1549.35,127,Encoragi,encoragi
Boia de Arqueamento Delta Zen Boost Vortex,4144.09,130,A N C O R A G E M,a n c o r a g e m
GPS AIS Zen,4984.15,8,E L E T R Ô N I C O S,e l e t r o n i c o s
GPS Simrad Zen Hydra Peak,10806.84,44,EletrônicoS,eletronicos


In [0]:
# Conferir categorias inválidas
valid_categories = ["eletronicos", "propulsao", "ancoragem"]

display(
    df_produtos_bronze
    .filter(~F.col("category").isin(valid_categories) | F.col("category").isNull())
    .groupBy("category")
    .count()
    .orderBy(F.desc("count"))
)

category,count
p r o p u l s a o,6
e l e t r o n i c o s,6
a n c o r a g e m,5
encoragem,5
encoragi,1


In [0]:
def normalize_category(category: str) -> str:
    """
    Padroniza categorias de produtos para:
    - eletronicos
    - propulsao
    - ancoragem

    Inclui tratamento de variações com erro de digitação e espaçamento.
    """
    if category is None:
        return None

    category_norm = normalize_text(category)
    category_norm = category_norm.replace(" ", "")

    # Eletrônicos
    if "eletr" in category_norm:
        return "eletronicos"

    # Propulsão
    if "prop" in category_norm:
        return "propulsao"

    # Ancoragem (incluindo erros comuns)
    if (
        "ancor" in category_norm
        or "encor" in category_norm
        or category_norm in [
            "ancoraguem",
            "ancorajm",
            "ancorajem",
            "ancorajen",
            "ancoragen",
            "encoragem",
            "encoragi"
        ]
    ):
        return "ancoragem"

    return None


normalize_category_udf = F.udf(normalize_category, StringType())

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/udf.py:103: UserWarning: Cannot infer the eval type from type hints. 
  warnings.warn("Cannot infer the eval type from type hints. ", UserWarning)


In [0]:
df_produtos_bronze = df_produtos_bronze.withColumn(
    "category",
    normalize_category_udf(F.col("actual_category"))
)

In [0]:
display(
    df_produtos_bronze
    .filter(F.col("category").isNull())
    .groupBy("actual_category")
    .count()
    .orderBy(F.desc("count"), F.asc("actual_category"))
)

actual_category,count


In [0]:
display(
    df_produtos_bronze
    .groupBy("category")
    .count()
    .orderBy(F.desc("count"))
)

category,count
ancoragem,53
propulsao,53
eletronicos,51


In [0]:
display(
    df_produtos_bronze
    .select("actual_category", "category")
    .distinct()
    .orderBy("actual_category")
)

actual_category,category
A N C O R A G E M,ancoragem
ANCORAGEM,ancoragem
AnCoRaGeM,ancoragem
AncorageM,ancoragem
Ancoragem,ancoragem
Ancoragen,ancoragem
Ancoraguem,ancoragem
AncorajeM,ancoragem
Ancorajem,ancoragem
Ancorajen,ancoragem


In [0]:
# Renomear colunas
df_produtos_bronze = df_produtos_bronze.withColumnRenamed("code", "product_id")
df_produtos_bronze = df_produtos_bronze.withColumnRenamed("name", "product_name")

In [0]:
# ======================================================================================
# ESCRITA - PRODUTOS BRONZE
# ======================================================================================

df_produtos_bronze.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .format("delta") \
    .saveAsTable("lh_nautical.02_bronze.produtos_bronze")

In [0]:
display(
    spark.table("lh_nautical.02_bronze.produtos_bronze")
)

product_name,price,product_id,actual_category,category
Rádio Furuno Zen Swift,35811.08,14,Eletroniscos,eletronicos
Sonda Lowrance Tidal Storm Vox,16432.35,19,eletronicos,eletronicos
Motor de Popa Volvo Hydro Dash 256HP,99664.64,72,Propução,propulsao
Motor Elétrico Honda Mako Axis 131HP,121761.56,78,propulsão,propulsao
Motor de Popa Parsun Impulse 162HP,38578.7,85,P R O P U L S Ã O,propulsao
Corrente Danforth Zen Abyss Axis,4103.01,117,ANCORAGEM,ancoragem
Cabo de Nylon Delta Velocity Core Mako,1549.35,127,Encoragi,ancoragem
Boia de Arqueamento Delta Zen Boost Vortex,4144.09,130,A N C O R A G E M,ancoragem
GPS AIS Zen,4984.15,8,E L E T R Ô N I C O S,eletronicos
GPS Simrad Zen Hydra Peak,10806.84,44,EletrônicoS,eletronicos


## 2. Transformação da tabela de custos de importação

Nesta etapa será realizado o tratamento estrutural do JSON aninhado, transformando o campo `historic_data` em múltiplas linhas.

Saída esperada:
- uma linha por produto, por data de vigência de custo, contendo o valor em dólar

In [0]:
# ======================================================================================
# LEITURA - CUSTOS DE IMPORTAÇÃO RAW
# ======================================================================================

df_importacao_raw = spark.table("lh_nautical.01_raw.custos_importacao_raw")
print("Quantidade de registros em custos_importacao_raw:", df_importacao_raw.count())
display(df_importacao_raw.limit(10))

Quantidade de registros em custos_importacao_raw: 150


category,historic_data,product_id,product_name
eletrônicos,"List(List(10/08/2016, 10583.63), List(15/06/2018, 8778.36), List(25/09/2018, 8023.87), List(19/03/2019, 8772.78), List(17/01/2020, 7918.18), List(17/06/2020, 6310.01), List(02/07/2021, 6586.7), List(16/05/2022, 6538.2), List(28/02/2023, 6360.91), List(17/10/2023, 6574.8), List(16/02/2024, 6657.12), List(22/02/2024, 6703.2), List(15/03/2024, 6633.66), List(02/08/2024, 5774.5), List(08/04/2025, 5579.75))",1,Transponder AIS Maré Magnum
eletrônicos,"List(List(23/11/2017, 4325.09), List(06/07/2022, 2577.22), List(24/05/2023, 2829.74), List(05/11/2025, 2602.27))",2,Transponder Furuno Marlin
eletrônicos,"List(List(12/04/2016, 2549.21), List(26/07/2018, 2423.45), List(27/06/2019, 2335.69), List(17/07/2019, 2399.28), List(14/10/2019, 2187.31), List(04/12/2020, 1745.49), List(21/06/2022, 1753.77))",3,Radar Furuno Pulse Leviathan
eletrônicos,"List(List(04/03/2016, 909.55), List(22/11/2016, 1010.42), List(19/04/2017, 1080.89), List(22/11/2018, 887.7), List(21/09/2022, 654.31), List(28/11/2023, 692.14), List(13/03/2024, 679.13), List(10/03/2025, 583.85))",4,Rádio AIS Hydro Tidal Zen
eletrônicos,"List(List(10/02/2016, 6006.45), List(17/01/2017, 7374.9), List(18/02/2021, 4364.4), List(03/05/2022, 4718.61), List(24/05/2022, 4920.79), List(15/08/2023, 4752.24), List(24/09/2024, 4327.37), List(24/07/2025, 4285.3))",5,Piloto Automático Furuno Storm
eletrônicos,"List(List(23/07/2020, 2288.92), List(01/12/2021, 2104.66), List(07/03/2024, 2394.79), List(21/06/2024, 2172.43), List(04/12/2024, 1951.33), List(19/03/2025, 2086.5))",6,Transponder AIS Vector
eletrônicos,"List(List(26/10/2016, 6253.41), List(21/06/2017, 5871.9), List(08/05/2018, 5454.91), List(12/12/2022, 3678.62), List(18/08/2025, 3604.64))",7,Radar AIS Zen
eletrônicos,"List(List(06/12/2019, 1193.04), List(19/06/2020, 932.31), List(18/02/2021, 919.04), List(06/08/2021, 951.1), List(23/03/2023, 947.09), List(10/05/2023, 1006.07), List(16/04/2024, 947.05), List(03/06/2025, 879.22))",8,GPS AIS Zen
eletrônicos,"List(List(26/12/2018, 10115.54), List(21/05/2020, 7088.62), List(10/12/2020, 7808.97), List(05/05/2022, 7933.96), List(16/04/2024, 7544.56), List(21/08/2024, 7258.64))",9,Transponder AIS Titan Pulse
eletrônicos,"List(List(29/05/2018, 8591.86), List(16/11/2021, 5849.07), List(20/12/2021, 5615.0), List(23/05/2023, 6449.43), List(29/06/2023, 6594.15), List(05/09/2023, 6445.41), List(13/11/2023, 6505.49), List(24/11/2023, 6547.91), List(07/10/2024, 5864.71), List(21/03/2025, 5596.76), List(01/10/2025, 6021.02))",10,Piloto Automático Simrad Titan Flux Magnum


In [0]:
# ======================================================================================
# TRANSFORMAÇÃO - CUSTOS DE IMPORTAÇÃO BRONZE
# ======================================================================================
# Objetivo:
# - padronizar nomes de colunas
# - explodir o array historic_data
# - selecionar colunas finais da estrutura normalizada
# - converter tipos de dados
# - tratar datas no formato brasileiro (dd/MM/yyyy)
# - remover registros críticos nulos
# ======================================================================================

df_importacao_raw = standardize_dataframe_columns(df_importacao_raw)

# Explode do array histórico para obter uma linha por produto e data de vigência
df_importacao_exploded = df_importacao_raw.withColumn(
    "historic_record",
    F.explode("historic_data")
)

# Seleção e tipagem das colunas finais
df_importacao_bronze = df_importacao_exploded.select(
    F.col("product_id").cast(IntegerType()).alias("product_id"),
    F.col("product_name").alias("product_name"),
    normalize_category_udf(F.col("category")).alias("category"),
    F.to_date(F.col("historic_record.start_date"), "dd/MM/yyyy").alias("start_date"),
    F.col("historic_record.usd_price").cast(DoubleType()).alias("usd_price")
)

# Remoção de registros com campos essenciais nulos
df_importacao_bronze = df_importacao_bronze.dropna(
    subset=["product_id", "start_date", "usd_price"]
)

# Visualização de amostra e contagem final
display(df_importacao_bronze.limit(10))
print("Quantidade de registros em custos_importacao_bronze:", df_importacao_bronze.count())

product_id,product_name,category,start_date,usd_price
1,Transponder AIS Maré Magnum,eletronicos,2016-08-10,10583.63
1,Transponder AIS Maré Magnum,eletronicos,2018-06-15,8778.36
1,Transponder AIS Maré Magnum,eletronicos,2018-09-25,8023.87
1,Transponder AIS Maré Magnum,eletronicos,2019-03-19,8772.78
1,Transponder AIS Maré Magnum,eletronicos,2020-01-17,7918.18
1,Transponder AIS Maré Magnum,eletronicos,2020-06-17,6310.01
1,Transponder AIS Maré Magnum,eletronicos,2021-07-02,6586.7
1,Transponder AIS Maré Magnum,eletronicos,2022-05-16,6538.2
1,Transponder AIS Maré Magnum,eletronicos,2023-02-28,6360.91
1,Transponder AIS Maré Magnum,eletronicos,2023-10-17,6574.8


Quantidade de registros em custos_importacao_bronze: 1260


In [0]:
# ======================================================================================
# ESCRITA - CUSTOS DE IMPORTAÇÃO BRONZE
# ======================================================================================

df_importacao_bronze.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable("lh_nautical.02_bronze.custos_importacao_bronze")

## 3. Transformação da tabela de vendas

Nesta etapa serão aplicados tratamentos técnicos mínimos:
- padronização de colunas
- ajuste de tipos numéricos
- conversão do campo de data
- preservação da granularidade transacional

In [0]:
# ======================================================================================
# LEITURA - VENDAS RAW
# ======================================================================================

df_vendas_raw = spark.table("lh_nautical.01_raw.vendas_raw")
print("Quantidade de registros em vendas_raw:", df_vendas_raw.count())
display(df_vendas_raw.limit(10))

Quantidade de registros em vendas_raw: 9895


id,id_client,id_product,qtd,total,sale_date
0,42,105,11,3405.0,2023-09-10
1,3,136,9,16873.9,15-09-2024
2,25,139,7,9475.3,2024-08-13
4,20,23,5,55893.0,2023-02-03
5,8,57,4,451403.9,2024-02-12
6,36,52,3,39056.4,2023-09-26
8,27,25,3,34560.05,2024-02-28
9,37,26,7,114932.9,07-11-2023
10,31,143,3,12643.55,2024-08-25
11,39,128,5,23254.0,2023-05-07


In [0]:
# ======================================================================================
# INSPEÇÃO - VENDAS RAW
# ======================================================================================
# Esta célula ajuda a confirmar nomes e tipos antes da transformação.
# ======================================================================================

df_vendas_raw.printSchema()

root
 |-- id: integer (nullable = true)
 |-- id_client: integer (nullable = true)
 |-- id_product: integer (nullable = true)
 |-- qtd: integer (nullable = true)
 |-- total: double (nullable = true)
 |-- sale_date: string (nullable = true)



In [0]:
# ======================================================================================
# TRANSFORMAÇÃO - VENDAS BRONZE
# ======================================================================================
# Objetivo:
# - padronizar nomes de colunas
# - converter colunas de data com suporte a múltiplos formatos
# - limpar campos monetários antes da conversão numérica
# - converter ids, quantidades e valores
# - padronizar nomenclatura para modelo analítico
# - manter a granularidade original da venda
# ======================================================================================

df_vendas_bronze = standardize_dataframe_columns(df_vendas_raw)

# --------------------------------------------------------------------------------------
# Conversão condicional de colunas de data (robusta)
# Suporta:
# - yyyy-MM-dd
# - dd/MM/yyyy
# - dd-MM-yyyy
# --------------------------------------------------------------------------------------
date_columns_candidates = ["sale_date", "date", "order_date", "data"]

for col_name in date_columns_candidates:
    if col_name in df_vendas_bronze.columns:
        df_vendas_bronze = df_vendas_bronze.withColumn(
            col_name,
            F.coalesce(
                F.expr(f"try_to_date({col_name}, 'yyyy-MM-dd')"),
                F.expr(f"try_to_date({col_name}, 'dd/MM/yyyy')"),
                F.expr(f"try_to_date({col_name}, 'dd-MM-yyyy')")
            )
        )

# --------------------------------------------------------------------------------------
# Limpeza de colunas monetárias antes do cast
# Remove símbolo monetário (R$), espaços e padroniza separador decimal
# --------------------------------------------------------------------------------------
monetary_columns = ["unit_price", "total"]

for col_name in monetary_columns:
    if col_name in df_vendas_bronze.columns:
        df_vendas_bronze = df_vendas_bronze.withColumn(
            col_name,
            F.regexp_replace(F.col(col_name).cast("string"), r"R\$\s*", "")
        )
        df_vendas_bronze = df_vendas_bronze.withColumn(
            col_name,
            F.regexp_replace(F.col(col_name), ",", ".")
        )

# --------------------------------------------------------------------------------------
# Conversão condicional de colunas numéricas
# --------------------------------------------------------------------------------------
numeric_columns_candidates = {
    "id": IntegerType(),
    "sale_id": IntegerType(),
    "product_id": IntegerType(),
    "id_product": IntegerType(),
    "client_id": IntegerType(),
    "id_client": IntegerType(),
    "customer_id": IntegerType(),
    "quantity": IntegerType(),
    "qtd": IntegerType(),
    "qty": IntegerType(),
    "unit_price": DoubleType(),
    "total": DoubleType()
}

for col_name, col_type in numeric_columns_candidates.items():
    if col_name in df_vendas_bronze.columns:
        df_vendas_bronze = df_vendas_bronze.withColumn(
            col_name,
            F.col(col_name).cast(col_type)
        )

# --------------------------------------------------------------------------------------
# Padronização de nomes de colunas (ESSENCIAL PARA MODELAGEM)
# --------------------------------------------------------------------------------------
rename_mapping = {
    "id": "sale_id",
    "id_client": "customer_id",
    "client_id": "customer_id",
    "id_product": "product_id",
    "qtd": "quantity",
    "qty": "quantity",
    "total": "total_amount"
}

for old_col, new_col in rename_mapping.items():
    if old_col in df_vendas_bronze.columns:
        df_vendas_bronze = df_vendas_bronze.withColumnRenamed(old_col, new_col)

# --------------------------------------------------------------------------------------
# Visualização inicial para validação
# --------------------------------------------------------------------------------------
display(df_vendas_bronze.limit(10))

sale_id,customer_id,product_id,quantity,total_amount,sale_date
0,42,105,11,3405.0,2023-09-10
1,3,136,9,16873.9,2024-09-15
2,25,139,7,9475.3,2024-08-13
4,20,23,5,55893.0,2023-02-03
5,8,57,4,451403.9,2024-02-12
6,36,52,3,39056.4,2023-09-26
8,27,25,3,34560.05,2024-02-28
9,37,26,7,114932.9,2023-11-07
10,31,143,3,12643.55,2024-08-25
11,39,128,5,23254.0,2023-05-07


In [0]:
# ======================================================================================
# ESCRITA - VENDAS BRONZE
# ======================================================================================

df_vendas_bronze.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable("lh_nautical.02_bronze.vendas_bronze")

## 4. Transformação da tabela de clientes

Nesta etapa serão aplicados tratamentos técnicos mínimos:
- padronização de colunas
- ajuste de tipos
- preservação da estrutura principal da entidade cliente

In [0]:
# ======================================================================================
# LEITURA - CLIENTES RAW
# ======================================================================================

df_clientes_raw = spark.table("lh_nautical.01_raw.clientes_raw")
print("Quantidade de registros em clientes_raw:", df_clientes_raw.count())
display(df_clientes_raw.limit(10))

Quantidade de registros em clientes_raw: 49


code,email,full_name,location
1,femininos.oliveira.antunes@icloud.com,Femininos Oliveira Antunes,"Aratu (Candeias) , BA"
2,nunes.fernanda.soares.azevedo.vieira@outlook.com,Fernanda Azevedo Soares Nunes Vieira,"PE , Recife"
3,farias.teixeira.daniel.ribeiro#gmail.com,Daniel Farias Ribeiro Teixeira,"Rio Grande,RS"
4,thiago.moreira#gmail.com,Thiago Moreira,"AC , Rio Branco"
5,pedro.freitas#icloud.com,Pedro Freitas,PA - Santarém Novo
6,coelho.pinheiro.peixoto.antônia.cavalcanti@aol.com,Antônia Coelho Pinheiro Peixoto Cavalcanti,"Fortaleza do Tabocão , TO"
7,torres.barros.rocha.bianca.siqueira#aol.com,Bianca Barros Rocha Torres Siqueira,PB/Cabedelo
8,pimentel.alves.luiz#outlook.com,Luiz Alves Pimentel,SE - Aracaju
9,lucas.lopes.guedes.cunha#tutanota.com,Lucas Guedes Cunha Lopes,PB - João Pessoa
10,paiva.débora#gmx.com,Débora Paiva,Santarém / PA


In [0]:
# ======================================================================================
# INSPEÇÃO - CLIENTES RAW
# ======================================================================================

df_clientes_raw.printSchema()

root
 |-- code: long (nullable = true)
 |-- email: string (nullable = true)
 |-- full_name: string (nullable = true)
 |-- location: string (nullable = true)



In [0]:
# ======================================================================================
# TRANSFORMAÇÃO - CLIENTES BRONZE
# ======================================================================================
# Objetivo:
# - padronizar nomes de colunas
# - renomear a chave de negócio para padrão analítico
# - higienizar campos textuais essenciais
# - corrigir e-mails com erro estrutural simples
# - remover duplicidades exatas
# - manter a integridade da entidade cliente
# ======================================================================================

df_clientes_bronze = standardize_dataframe_columns(df_clientes_raw)

# --------------------------------------------------------------------------------------
# Padronização de nomenclatura para modelo analítico
# --------------------------------------------------------------------------------------
rename_mapping = {
    "code": "customer_id",
    "full_name": "customer_name"
}

for old_col, new_col in rename_mapping.items():
    if old_col in df_clientes_bronze.columns:
        df_clientes_bronze = df_clientes_bronze.withColumnRenamed(old_col, new_col)

# --------------------------------------------------------------------------------------
# Conversão condicional de colunas inteiras comuns
# --------------------------------------------------------------------------------------
client_integer_columns = ["customer_id", "client_id", "age"]

for col_name in client_integer_columns:
    if col_name in df_clientes_bronze.columns:
        df_clientes_bronze = df_clientes_bronze.withColumn(
            col_name,
            F.col(col_name).cast(IntegerType())
        )

# --------------------------------------------------------------------------------------
# Conversão condicional de colunas de data comuns
# --------------------------------------------------------------------------------------
client_date_columns = ["created_at", "signup_date", "birth_date", "date_of_birth"]

for col_name in client_date_columns:
    if col_name in df_clientes_bronze.columns:
        df_clientes_bronze = df_clientes_bronze.withColumn(
            col_name,
            F.coalesce(
                F.expr(f"try_to_date({col_name}, 'yyyy-MM-dd')"),
                F.expr(f"try_to_date({col_name}, 'dd/MM/yyyy')"),
                F.expr(f"try_to_date({col_name}, 'dd-MM-yyyy')")
            )
        )

# --------------------------------------------------------------------------------------
# Higienização de e-mail
# Corrige erro simples identificado na base: uso de "#" no lugar de "@"
# --------------------------------------------------------------------------------------
if "email" in df_clientes_bronze.columns:
    df_clientes_bronze = df_clientes_bronze.withColumn(
        "email",
        F.lower(F.trim(F.col("email")))
    )

    df_clientes_bronze = df_clientes_bronze.withColumn(
        "email",
        F.regexp_replace(F.col("email"), "#", "@")
    )

# --------------------------------------------------------------------------------------
# Higienização de nome do cliente
# --------------------------------------------------------------------------------------
if "customer_name" in df_clientes_bronze.columns:
    df_clientes_bronze = df_clientes_bronze.withColumn(
        "customer_name",
        F.trim(F.col("customer_name"))
    )

# --------------------------------------------------------------------------------------
# Higienização básica de localização
# Mantém o campo textual, mas remove excesso de espaços
# --------------------------------------------------------------------------------------
if "location" in df_clientes_bronze.columns:
    df_clientes_bronze = df_clientes_bronze.withColumn(
        "location",
        F.trim(F.regexp_replace(F.col("location"), r"\s+", " "))
    )

# --------------------------------------------------------------------------------------
# Remoção de duplicidades exatas
# --------------------------------------------------------------------------------------
total_clientes_antes = df_clientes_bronze.count()

df_clientes_bronze = df_clientes_bronze.dropDuplicates()

total_clientes_depois = df_clientes_bronze.count()
duplicados_removidos_clientes = total_clientes_antes - total_clientes_depois

print("Duplicados removidos em clientes:", duplicados_removidos_clientes)

# --------------------------------------------------------------------------------------
# Visualização inicial para validação
# --------------------------------------------------------------------------------------
display(df_clientes_bronze.limit(50))

Duplicados removidos em clientes: 0


customer_id,email,customer_name,location
1,femininos.oliveira.antunes@icloud.com,Femininos Oliveira Antunes,"Aratu (Candeias) , BA"
2,nunes.fernanda.soares.azevedo.vieira@outlook.com,Fernanda Azevedo Soares Nunes Vieira,"PE , Recife"
3,farias.teixeira.daniel.ribeiro@gmail.com,Daniel Farias Ribeiro Teixeira,"Rio Grande,RS"
4,thiago.moreira@gmail.com,Thiago Moreira,"AC , Rio Branco"
5,pedro.freitas@icloud.com,Pedro Freitas,PA - Santarém Novo
6,coelho.pinheiro.peixoto.antônia.cavalcanti@aol.com,Antônia Coelho Pinheiro Peixoto Cavalcanti,"Fortaleza do Tabocão , TO"
7,torres.barros.rocha.bianca.siqueira@aol.com,Bianca Barros Rocha Torres Siqueira,PB/Cabedelo
8,pimentel.alves.luiz@outlook.com,Luiz Alves Pimentel,SE - Aracaju
9,lucas.lopes.guedes.cunha@tutanota.com,Lucas Guedes Cunha Lopes,PB - João Pessoa
10,paiva.débora@gmx.com,Débora Paiva,Santarém / PA


In [0]:
display(
    df_clientes_bronze.filter(~F.col("email").contains("@"))
)

customer_id,email,customer_name,location


In [0]:
df_clientes_bronze.printSchema()

root
 |-- customer_id: integer (nullable = true)
 |-- email: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- location: string (nullable = true)



In [0]:
display(
    df_clientes_bronze.filter(F.col("customer_id").isNull())
)

customer_id,email,customer_name,location


In [0]:
# ======================================================================================
# ESCRITA - CLIENTES BRONZE
# ======================================================================================

df_clientes_bronze.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable("lh_nautical.02_bronze.clientes_bronze")

## 5. Validações finais da camada bronze

Nesta etapa são verificadas:
- quantidade de registros por tabela
- persistência correta no catálogo
- disponibilidade para consumo posterior em dbt

In [0]:
# ======================================================================================
# VALIDAÇÃO FINAL - CONTAGENS
# ======================================================================================

spark.sql("SELECT COUNT(*) AS total FROM lh_nautical.02_bronze.vendas_bronze").show()
spark.sql("SELECT COUNT(*) AS total FROM lh_nautical.02_bronze.produtos_bronze").show()
spark.sql("SELECT COUNT(*) AS total FROM lh_nautical.02_bronze.clientes_bronze").show()
spark.sql("SELECT COUNT(*) AS total FROM lh_nautical.02_bronze.custos_importacao_bronze").show()

+-----+
|total|
+-----+
| 9895|
+-----+

+-----+
|total|
+-----+
|  157|
+-----+

+-----+
|total|
+-----+
|   49|
+-----+

+-----+
|total|
+-----+
| 1260|
+-----+



In [0]:
# VALIDAÇÃO FINAL - AMOSTRAS DAS TABELAS BRONZE


tables = [
    "vendas_bronze",
    "produtos_bronze",
    "clientes_bronze",
    "custos_importacao_bronze"
]

for table in tables:
    print(f"\n🔹 TABELA: {table}")
    display(
        spark.table(f"lh_nautical.02_bronze.{table}").limit(5)
    )


🔹 TABELA: vendas_bronze


sale_id,customer_id,product_id,quantity,total_amount,sale_date
0,42,105,11,3405.0,2023-09-10
1,3,136,9,16873.9,2024-09-15
2,25,139,7,9475.3,2024-08-13
4,20,23,5,55893.0,2023-02-03
5,8,57,4,451403.9,2024-02-12



🔹 TABELA: produtos_bronze


product_name,price,product_id,actual_category,category
Rádio Furuno Zen Swift,35811.08,14,Eletroniscos,eletronicos
Sonda Lowrance Tidal Storm Vox,16432.35,19,eletronicos,eletronicos
Motor de Popa Volvo Hydro Dash 256HP,99664.64,72,Propução,propulsao
Motor Elétrico Honda Mako Axis 131HP,121761.56,78,propulsão,propulsao
Motor de Popa Parsun Impulse 162HP,38578.7,85,P R O P U L S Ã O,propulsao



🔹 TABELA: clientes_bronze


customer_id,email,customer_name,location
5,pedro.freitas@icloud.com,Pedro Freitas,PA - Santarém Novo
12,rafael.pereira.barros@zoho.com,Rafael Pereira Barros,"TO , Fortaleza do Tabocão"
41,costa.correia.ribeiro.santos.pedro@yahoo.com,Pedro Ribeiro Costa Santos Correia,BA / Aratu (Candeias)
7,torres.barros.rocha.bianca.siqueira@aol.com,Bianca Barros Rocha Torres Siqueira,PB/Cabedelo
8,pimentel.alves.luiz@outlook.com,Luiz Alves Pimentel,SE - Aracaju



🔹 TABELA: custos_importacao_bronze


product_id,product_name,category,start_date,usd_price
1,Transponder AIS Maré Magnum,eletronicos,2016-08-10,10583.63
1,Transponder AIS Maré Magnum,eletronicos,2018-06-15,8778.36
1,Transponder AIS Maré Magnum,eletronicos,2018-09-25,8023.87
1,Transponder AIS Maré Magnum,eletronicos,2019-03-19,8772.78
1,Transponder AIS Maré Magnum,eletronicos,2020-01-17,7918.18


## Conclusão

A camada `02_bronze` foi construída para as quatro fontes do projeto, preservando a rastreabilidade dos dados e preparando o ambiente para a camada `03_silver`.

### Próximos passos
- criar modelos `dbt` sobre as tabelas bronze
- consolidar entidades e regras de negócio na camada silver
- construir tabelas analíticas finais na camada gold 